# Data Exploration

Explore the WLASL dataset before building the preprocessing pipeline.

In [12]:
import json

with open('../raw_data/WLASL_v0.3.json') as f:
    data = json.load(f)

print(f"Loaded {len(data)} entries")

Loaded 2000 entries


In [13]:
import json
import pandas as pd
# Cell 2 — See how many videos exist per word
word_counts = []
for entry in data:
    word_counts.append({
        'word': entry['gloss'],
        'num_videos': len(entry['instances'])
    })

df = pd.DataFrame(word_counts)
print(df.sort_values('num_videos', ascending=False).head(20))
# This shows you the 20 words with the most training videos
# These are the words you should train on first

print(f"Total unique words: {len(data)}")

        word  num_videos
0       book          40
1      drink          35
2   computer          30
3     before          26
4      chair          26
5         go          26
6    clothes          25
7        who          25
8      candy          24
9     cousin          23
10      deaf          23
15      walk          22
17       yes          22
16      year          22
12      help          22
14      thin          22
13        no          22
11      fine          22
25    mother          21
30      what          21
Total unique words: 2000


In [14]:
# Cell 3 — Open one video and look at it
import cv2

video_path = '../raw_data/videos/00335.mp4'
cap = cv2.VideoCapture(video_path)

print(f"Total frames: {cap.get(cv2.CAP_PROP_FRAME_COUNT)}")
print(f"FPS: {cap.get(cv2.CAP_PROP_FPS)}")
print(f"Width: {cap.get(cv2.CAP_PROP_FRAME_WIDTH)}")
print(f"Height: {cap.get(cv2.CAP_PROP_FRAME_HEIGHT)}")

cap.release()

Total frames: 58.0
FPS: 25.0
Width: 320.0
Height: 240.0


In [15]:
# Cell 4 — Read one frame and run MediaPipe on it
# mediapipe 0.10+ removed mp.solutions — use the Tasks API instead
import mediapipe as mp
import mediapipe.tasks as tasks
import cv2
import numpy as np
import matplotlib.pyplot as plt
import subprocess, os

BaseOptions = tasks.BaseOptions
vision = tasks.vision

model_path = '../raw_data/hand_landmarker.task'
if not os.path.exists(model_path):
    url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
    print('Downloading hand landmarker model...')
    subprocess.run(['curl', '-L', '-o', model_path, url], check=True)
    print('Done.')

options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    num_hands=2,
    min_hand_detection_confidence=0.5
)

cap = cv2.VideoCapture('../raw_data/videos/00335.mp4')
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# Jump to the middle of the video — frame 0 is often before the signer starts
cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
ret, frame = cap.read()
cap.release()

rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

with vision.HandLandmarker.create_from_options(options) as landmarker:
    result = landmarker.detect(mp_image)

if result.hand_landmarks:
    print(f"Hands detected: {len(result.hand_landmarks)}")
    landmarks = result.hand_landmarks[0]
    print(f"Number of landmarks: {len(landmarks)}")
    print(f"First landmark x={landmarks[0].x:.4f} y={landmarks[0].y:.4f} z={landmarks[0].z:.4f}")
else:
    print("No hands detected in this frame — try a different video")

Hands detected: 1
Number of landmarks: 21
First landmark x=0.4118 y=0.8846 z=-0.0000


I0000 00:00:1777839527.633695 5224936 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4
W0000 00:00:1777839527.641525 5224940 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777839527.652091 5224940 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [16]:
# Raw landmarks are in image coordinates (0.0 to 1.0 of the frame)
# Problem: hand closer to camera = larger numbers, hand further away = smaller numbers
# Same sign looks completely different to the model

# The fix — normalize relative to the wrist (landmark 0)
# and scale by the size of the hand bounding box

def normalize_landmarks(landmarks):
    """
    Takes a list of 21 MediaPipe landmark objects
    Returns a flat numpy array of 63 normalized numbers
    """
    # Convert to numpy array — shape (21, 3)
    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
    
    # Step 1: Subtract the wrist position (landmark 0)
    # Now wrist is always at (0, 0, 0) regardless of where it appears in frame
    wrist = coords[0]
    coords = coords - wrist
    
    # Step 2: Scale by the hand size (distance from wrist to middle finger base)
    # This makes the model invariant to how far the person sits from the camera
    hand_size = np.linalg.norm(coords[9])  # landmark 9 = middle finger base
    if hand_size > 0:
        coords = coords / hand_size
    
    # Step 3: Flatten to 63 numbers (21 landmarks × 3 coordinates)
    return coords.flatten()

# Test it
if result.hand_landmarks:
    raw = result.hand_landmarks[0]  # Tasks API: already the list of 21 landmarks
    normalized = normalize_landmarks(raw)
    print(f"Output shape: {normalized.shape}")  # should be (63,)
    print(f"Wrist after normalization: {normalized[0:3]}")  # should be [0, 0, 0]
    print(f"Sample values: {normalized[3:9]}")

Output shape: (63,)
Wrist after normalization: [0. 0. 0.]
Sample values: [ 0.21698976 -0.59270866 -0.13652329  0.54508662 -0.91289317 -0.21888286]
